<a href="https://colab.research.google.com/github/bquast/colab/blob/master/sft_smollm2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers datasets trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.9 MB/s eta 0:00:00


In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

2.11.0+cu128
True
Tesla T4
14.6


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "HuggingFaceTB/SmolLM2-360M"

tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name,
    torch_dtype=torch.float16
).cuda()

print(sum(p.numel() for p in model.parameters()) / 1e6)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

361.82112


In [ ]:
prompt = "User: What causes inflation?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

User: What causes inflation?
Assistant: Inflation is caused by a decrease in the value of money.

Teacher: What is the value of money?
Assistant: The value of money is the amount of goods and services that can be bought with a certain amount of money.

Teacher: What is the value of goods and


In [ ]:
from datasets import load_dataset, Dataset

stream = load_dataset(
    "HuggingFaceTB/smol-smoltalk",
    split="train",
    streaming=True
)

data = Dataset.from_list(list(stream.take(500)))

def format_example(x):
    text = "\n".join(
        f"{m['role'].title()}: {m['content']}"
        for m in x["messages"]
    )
    return {"text": text + tokenizer.eos_token}

data = data.map(format_example)

print(len(data))
print(data[0]["text"][:1000])

README.md:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

500
User: I need you to edit something for me. This is the text I wrote, 

"Me and my friends have been waiting for a long time to go back to the movies and catch a movie we been waiting on. Last nite I finely went to go buy tickets and when I got to the movie theater the tickets were sold out, so I was pretty pist. We were all pist. So today we deside if we wanted to go to a game instead, but I dont think we will go to a game now."
Assistant: Here's a revised version of your text with some suggested edits to improve grammar, clarity, and overall flow:

"My friends and I have been waiting a long time to return to the movies and see a film we've been eagerly anticipating. Last night, I finally went to buy tickets, but when I arrived at the theater, they were sold out. I was pretty upset, and my friends were too. Today, we discussed the possibility of going to a game instead, but I don't think we'll end up going."

I made the following changes:

- Changed "Me and my friends" to "My frien

In [ ]:
data = data.select_columns(["text"])

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="smollm-sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    max_length=512,
    fp16=True,
    logging_steps=5,
    report_to="none"
)

del trainer, model
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    name,
    torch_dtype=torch.float32
).cuda()

print(next(model.parameters()).dtype)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data,
    processing_class=tokenizer
)

trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

torch.float32


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
5,1.590511
10,1.558562
15,1.459295
20,1.426375
25,1.397282
30,1.452783


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=32, training_loss=1.4868524000048637, metrics={'train_runtime': 299.6007, 'train_samples_per_second': 1.669, 'train_steps_per_second': 0.107, 'total_flos': 462883754995200.0, 'train_loss': 1.4868524000048637, 'entropy': 1.7172163546085357, 'num_tokens': 213667.0, 'mean_token_accuracy': 0.6073073238134384, 'epoch': 1.0})

In [ ]:
model.eval()

prompt = "User: What causes inflation?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

User: What causes inflation?
Assistant: Inflation occurs when the general price level of goods and services increases over time. This can be caused by various factors, such as increased demand for goods and services, increased production costs, or changes in the supply of money.

Inflation can have both positive and negative effects on the economy. On


In [ ]:
model.save_pretrained("smollm2-360m-sft")
tokenizer.save_pretrained("smollm2-360m-sft")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]